# BPR Recommender System - Kaggle Runner

This notebook runs the BPR (Bayesian Personalized Ranking) recommender system on Kaggle with GPU acceleration.

## Setup Instructions

1. **Enable GPU**: Settings → Accelerator → GPU T4 x2
2. **Add Dataset**: Add the MovieLens 32M dataset (either upload or use existing)
3. **Upload Code**: Upload your `recsys/` directory as a dataset or use git clone
4. **Run cells** sequentially

---

## 1. Setup and Installation

In [ ]:
# Install additional dependencies if needed
!pip install PyYAML -q

In [ ]:
# Copy code to working directory (if uploaded as dataset)
# Uncomment if you uploaded code as a Kaggle dataset
# !cp -r /kaggle/input/recsys-code/* /kaggle/working/

# OR clone from GitHub (if repo is public)
!git clone https://github.com/codetuscan/recsys.git
%cd recsys

## 2. Import and Check Environment

In [ ]:
import sys
import os

# Add recsys to path if needed
if '/kaggle/working/recsys' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Import modules
from recsys.utils import print_environment_info
from recsys.config import load_config
from recsys.experiments.experiment_runner import ExperimentRunner

# Check environment
print_environment_info()

## 3. Load Configuration

This automatically detects the Kaggle environment and loads appropriate settings.

In [ ]:
# Load Kaggle configuration
config = load_config('kaggle')

# You can modify config parameters here
# config.model.batch_size = 4096
# config.model.epochs = 10
# config.data.data_subset = 0.1  # Use 10% of data for testing

print(f"Experiment: {config.experiment.experiment_name}")
print(f"Batch size: {config.model.batch_size}")
print(f"Epochs: {config.model.epochs}")
print(f"Device: {config.experiment.device}")

## 4. Run Experiment

This cell runs the complete pipeline:
1. Load and preprocess data
2. Create model and dataloaders
3. Train with GPU acceleration
4. Evaluate and save results

In [ ]:
# Create experiment runner
runner = ExperimentRunner(config)

# Run experiment
results = runner.run()

## 5. View Results

In [ ]:
# Print final results
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
for key, value in results.items():
    if isinstance(value, float):
        print(f"{key:20s}: {value:.4f}")
    else:
        print(f"{key:20s}: {value}")
print("="*60)

## 6. Visualize Training Curves (Optional)

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training metrics
experiment_dir = runner.metrics_logger.get_experiment_dir()
train_metrics_path = experiment_dir / "train_metrics.json"
eval_metrics_path = experiment_dir / "eval_metrics.json"

with open(train_metrics_path) as f:
    train_metrics = json.load(f)

with open(eval_metrics_path) as f:
    eval_metrics = json.load(f)

# Plot training loss
epochs = [m['epoch'] for m in train_metrics]
losses = [m['loss'] for m in train_metrics]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

# Plot evaluation metrics
plt.subplot(1, 2, 2)
eval_epochs = [m['epoch'] for m in eval_metrics]
ndcg_10 = [m['ndcg@10'] for m in eval_metrics]
precision_10 = [m['precision@10'] for m in eval_metrics]

plt.plot(eval_epochs, ndcg_10, label='NDCG@10', marker='o')
plt.plot(eval_epochs, precision_10, label='Precision@10', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Evaluation Metrics')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training curves saved to /kaggle/working/training_curves.png")

## 7. Download Results

All outputs are saved to `/kaggle/working/outputs/` and will be available for download after the notebook completes.

In [ ]:
# List output files
!ls -lh /kaggle/working/outputs/

---

## Next Steps

1. **Download results**: All files in `/kaggle/working/outputs/` are available
2. **Try different hyperparameters**: Modify config in Cell 3 and re-run
3. **Extend the model**: Add new models or evaluation metrics
4. **Compare experiments**: Run multiple experiments with different settings

---